In [ ]:
%pip install iterative-stratification
%pip install ultralytics

In [ ]:
import torch
import shutil

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("CUDA is not available.")

import importlib

drive = importlib.import_module("google.colab.drive")

getattr(drive, "mount")("/content/drive")

from pathlib import Path

path_archive_drive = Path("/content/drive/MyDrive/indoor_object_detection.zip")
path_archive = Path("/content/indoor_object_detection.zip")

if not path_archive.exists():
    assert path_archive_drive.exists()

    shutil.copy2(path_archive_drive, path_archive)

path_root = Path("/content/indoor_object_detection")

import zipfile

with zipfile.ZipFile(path_archive) as zf:
    zf.extractall(path_root)

shutil.move(path_root / "Indoor Object Detection Dataset", path_root / "dataset")

path_root = path_root / "dataset"

annotations = [
    'annotation/annotation_s1.xml',
    'annotation/annotation_s2.xml',
    'annotation/annotation_s3.xml',
    'annotation/annotation_s4.xml',
    'annotation/annotation_s5.xml',
    'annotation/annotation_s6.xml',
]

sequences = [
    'sequence_1',
    'sequence_2',
    'sequence_3',
    'sequence_4',
    'sequence_5',
    'sequence_6',
]
sequences = list(map(Path, sequences))

annotations = [path_root / annotation for annotation in annotations]

In [ ]:
height = 720
width = 1280

split_ratios = {
    "train": 0.8,
    "val": 0.1,
    "test": 0.1
}
import os
print(sorted(os.listdir(path_root)))

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET


class BBox:
    def __init__(self, label: str, center_x: float, center_y: float, norm_width: float, norm_height: float):
        self.label = label
        self.center_x = center_x
        self.center_y = center_y
        self.norm_width = norm_width
        self.norm_height = norm_height

    def __repr__(self):
        return f"{self.center_x:.6f} {self.center_y:.6f} {self.norm_width:.6f} {self.norm_height:.6f}"


def convert_xml_to_boxes(
    xml_path: str | Path,
    image_height: int,
    image_width: int,
) -> dict[str, list[BBox]]:
    xml_path = Path(xml_path)

    tree = ET.parse(xml_path)
    root = tree.getroot()

    result: dict[str, list[BBox]] = {}

    for image_elem in root.findall("./images/image"):
        image_filename = image_elem.attrib["file"]

        boxes = []

        for box_elem in image_elem.findall("box"):
            top = float(box_elem.attrib["top"])
            left = float(box_elem.attrib["left"])
            width = float(box_elem.attrib["width"])
            height = float(box_elem.attrib["height"])

            label_elem = box_elem.find("label")
            if label_elem is None or label_elem.text is None:
                raise ValueError(
                    f"Missing label for box in image {image_filename}"
                )

            label = label_elem.text.strip()

            center_x = left + width / 2.0
            center_y = top + height / 2.0

            center_x /= image_width
            center_y /= image_height
            norm_width = width / image_width
            norm_height = height / image_height

            boxes.append(BBox(label, center_x, center_y, norm_width, norm_height))

        result[image_filename] = boxes

    return result

all_boxes = [convert_xml_to_boxes(annotation, height, width) for annotation in annotations]

In [ ]:
import json

all_labels: set[str] = set()

for boxes in all_boxes:
    for box_list in boxes.values():
        for box in box_list:
            all_labels.add(box.label)

all_labels = sorted(all_labels)

import numpy as np

label_ids = np.arange(len(all_labels)).tolist()
label_map = dict(zip(all_labels, label_ids))
label_map_reverse = dict(zip(label_ids, all_labels))

image_to_classes: dict[str, list[int]] = {}

for boxes in all_boxes:
    for image_filename, box_list in boxes.items():
        class_ids = set(label_map[box.label] for box in box_list)
        image_to_classes[image_filename] = sorted(class_ids)

print(json.dumps(label_map))

In [ ]:
image_to_sequence: dict[str, Path] = {}

for sequence, boxes in zip(sequences, all_boxes):
    for image_filename, box_list in boxes.items():
        image_path = path_root / sequence / image_filename

        if not image_path.exists():
            raise FileNotFoundError(f"Image file {image_path} does not exist")

        image_to_sequence[image_filename] = sequence

        label_path = image_path.with_suffix(".txt")

        with open(label_path, "w") as f:
            for box in box_list:
                f.write(f"{label_map[box.label]} {repr(box)}\n")

print(f"Total number of images: {len(image_to_sequence)}")

In [ ]:
import numpy as np
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit


def split_detection_dataset(
    image_to_classes: dict[str, list[int]],
    train_ratio: float = 0.8,
    val_ratio: float = 0.1,
    test_ratio: float = 0.1,
    random_state: int = 42,
) -> dict[str, list[str]]:
    """
    Split a detection dataset using multilabel stratification.

    Each image is associated with a list of unique class IDs present
    in that image.

    Returns:
        train_images, val_images, test_images
    """
    if not np.isclose(train_ratio + val_ratio + test_ratio, 1.0):
        raise ValueError("train_ratio + val_ratio + test_ratio must equal 1.0")

    image_paths = np.array(list(image_to_classes.keys()))

    all_class_ids = sorted({
        class_id
        for class_ids in image_to_classes.values()
        for class_id in class_ids
    })

    if not all_class_ids:
        raise ValueError("No class IDs found in image_to_classes")

    class_to_col = {
        class_id: col
        for col, class_id in enumerate(all_class_ids)
    }

    y = np.zeros(
        (len(image_paths), len(all_class_ids)),
        dtype=np.uint8,
    )

    for row, image_path in enumerate(image_paths):
        for class_id in image_to_classes[image_path]:
            y[row, class_to_col[class_id]] = 1

    remaining_ratio = val_ratio + test_ratio

    first_split = MultilabelStratifiedShuffleSplit(
        n_splits=1,
        test_size=remaining_ratio,
        random_state=random_state,
    )

    train_idx, remaining_idx = next(
        first_split.split(image_paths, y)
    )

    test_fraction_of_remaining = test_ratio / remaining_ratio

    second_split = MultilabelStratifiedShuffleSplit(
        n_splits=1,
        test_size=test_fraction_of_remaining,
        random_state=random_state,
    )

    val_relative_idx, test_relative_idx = next(
        second_split.split(
            image_paths[remaining_idx],
            y[remaining_idx],
        )
    )

    val_idx = remaining_idx[val_relative_idx]
    test_idx = remaining_idx[test_relative_idx]

    return {
        "train": image_paths[train_idx].tolist(),
        "val": image_paths[val_idx].tolist(),
        "test": image_paths[test_idx].tolist(),
    }


splits = split_detection_dataset(
    image_to_classes,
    train_ratio=split_ratios["train"],
    val_ratio=split_ratios["val"],
    test_ratio=split_ratios["test"],
)

class_appearances = {
    split: np.zeros(len(label_map), dtype=int) for split in splits.keys()
}

for split, image_list in splits.items():
    print(f"[{split}] split contains {len(image_list)} images")

    split_file = path_root / f"{split}.txt"

    with split_file.open("w") as f:
        for image_filename in image_list:
            class_appearances[split][image_to_classes[image_filename]] += 1

            f.write(f"./{image_to_sequence[image_filename] / image_filename}\n")

print()
print(f"Class distribution:")
for split, appearances in class_appearances.items():
    print(f"[{split}] {appearances.tolist()}")

In [ ]:
import numpy as np
from PIL import Image
from matplotlib import pyplot as plt

random_state = np.random.default_rng(42)

image_filename = random_state.choice(splits["train"])

image_path = path_root / image_to_sequence[image_filename] / image_filename

if not isinstance(image_path, Path) or not image_path.exists():
    raise FileNotFoundError(f"Image file {image_path} does not exist")

label_path = image_path.with_suffix(".txt")

if not label_path.exists():
    raise FileNotFoundError(f"Label file {label_path} does not exist")

image = Image.open(image_path)

boxes: list[BBox] = []

for line in label_path.read_text().splitlines():
    class_id, center_x, center_y, norm_width, norm_height = line.split()

    boxes.append(
        BBox(
            label=label_map_reverse[int(class_id)],
            center_x=float(center_x),
            center_y=float(center_y),
            norm_width=float(norm_width),
            norm_height=float(norm_height),
        )
    )

figure = plt.figure(figsize=(12, 8))

ax = figure.add_subplot(1, 1, 1)

ax.imshow(image)
ax.set_title(image_filename)

for box in boxes:
    width, height = image.size

    x = (box.center_x - box.norm_width / 2) * width
    y = (box.center_y - box.norm_height / 2) * height
    w = box.norm_width * width
    h = box.norm_height * height

    ax.add_patch(
        plt.Rectangle(
            (x, y),
            w,
            h,
            linewidth=2,
            edgecolor="red",
            facecolor="none",
        )
    )

figure.show()

In [ ]:
import yaml

metadata = {
    "path": str(path_root),
    "train": "train.txt",
    "val": "val.txt",
    "test": "test.txt",
    "names": label_map_reverse,
}

path_metadata = path_root / "metadata.yaml"

with open(path_metadata, "w") as f:
    yaml.safe_dump(metadata, f, sort_keys=False)

In [ ]:
from ultralytics.data.dataset import YOLODataset
from ultralytics.data.utils import check_det_dataset

path_metadata = path_root / "metadata.yaml"

data = check_det_dataset(path_metadata)

datasets = {
    split: YOLODataset(
        img_path=data[split],
        data=data,
        task="detect",
        imgsz=640,
    ) for split in splits.keys()
}

for key in datasets.keys():
    print(f"Number of {key} images: {len(datasets[key])}")

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26s.pt")

results = model.train(
    data=path_metadata,
    epochs=100,
    imgsz=640,
    patience=20,
    lr0=1e-3,
    lrf=1e-2,
    project="/content/drive/MyDrive/DocuSketch",
    name="yolo26s_res640",
)

In [ ]:
from pathlib import Path

import cv2
import numpy as np


def load_yolo_ground_truth(
    image_path: Path,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Load YOLO-format GT labels and convert normalized xywh boxes
    into pixel-coordinate xyxy boxes.

    Returns:
        gt_boxes:
            Float array with shape (N, 4), in pixel xyxy format.

        gt_classes:
            Integer array with shape (N,).
    """
    image = cv2.imread(str(image_path))

    if image is None:
        raise RuntimeError(f"Could not read image: {image_path}")

    image_height, image_width = image.shape[:2]
    label_path = image_path.with_suffix(".txt")

    if not label_path.exists():
        return (
            np.empty((0, 4), dtype=np.float32),
            np.empty((0,), dtype=np.int64),
        )

    gt_boxes = []
    gt_classes = []

    with label_path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            fields = line.strip().split()

            if not fields:
                continue

            if len(fields) != 5:
                raise ValueError(
                    f"{label_path}:{line_number}: "
                    f"expected 5 values, received {len(fields)}"
                )

            class_id, center_x, center_y, width, height = map(
                float,
                fields,
            )

            x1 = (center_x - width / 2.0) * image_width
            y1 = (center_y - height / 2.0) * image_height
            x2 = (center_x + width / 2.0) * image_width
            y2 = (center_y + height / 2.0) * image_height

            gt_boxes.append([x1, y1, x2, y2])
            gt_classes.append(int(class_id))

    return (
        np.asarray(gt_boxes, dtype=np.float32).reshape(-1, 4),
        np.asarray(gt_classes, dtype=np.int64),
    )


def pairwise_iou(
    boxes_a: np.ndarray,
    boxes_b: np.ndarray,
) -> np.ndarray:
    """
    Compute all pairwise IoUs between two arrays of xyxy boxes.

    Args:
        boxes_a: shape (N, 4)
        boxes_b: shape (M, 4)

    Returns:
        IoU matrix with shape (N, M).
    """
    if len(boxes_a) == 0 or len(boxes_b) == 0:
        return np.zeros(
            (len(boxes_a), len(boxes_b)),
            dtype=np.float32,
        )

    intersection_top_left = np.maximum(
        boxes_a[:, None, :2],
        boxes_b[None, :, :2],
    )

    intersection_bottom_right = np.minimum(
        boxes_a[:, None, 2:],
        boxes_b[None, :, 2:],
    )

    intersection_size = np.clip(
        intersection_bottom_right - intersection_top_left,
        a_min=0.0,
        a_max=None,
    )

    intersection_area = (
        intersection_size[..., 0]
        * intersection_size[..., 1]
    )

    area_a = (
        np.clip(boxes_a[:, 2] - boxes_a[:, 0], 0.0, None)
        * np.clip(boxes_a[:, 3] - boxes_a[:, 1], 0.0, None)
    )

    area_b = (
        np.clip(boxes_b[:, 2] - boxes_b[:, 0], 0.0, None)
        * np.clip(boxes_b[:, 3] - boxes_b[:, 1], 0.0, None)
    )

    union_area = (
        area_a[:, None]
        + area_b[None, :]
        - intersection_area
    )

    return intersection_area / np.clip(
        union_area,
        a_min=1e-9,
        a_max=None,
    )


def match_boxes_by_iou(
    pred_boxes: np.ndarray,
    gt_boxes: np.ndarray,
) -> dict:
    """
    Greedily perform one-to-one matching using the largest remaining IoU.

    This function receives boxes from only one class, so class checking
    is not required here.

    Every GT contributes to mean_gt_iou:
        - matched GT: matched IoU;
        - unmatched GT: IoU 0.

    Predictions are allowed to match GT boxes even when IoU is low,
    because the goal is to assess the best available localization.
    """
    num_predictions = len(pred_boxes)
    num_gt = len(gt_boxes)

    if num_gt == 0:
        raise ValueError(
            "match_boxes_by_iou() requires at least one GT box."
        )

    iou_matrix = pairwise_iou(pred_boxes, gt_boxes)

    unmatched_prediction_indices = set(range(num_predictions))
    unmatched_gt_indices = set(range(num_gt))

    matches = []

    while unmatched_prediction_indices and unmatched_gt_indices:
        best_prediction_index = None
        best_gt_index = None
        best_iou = -1.0

        for prediction_index in unmatched_prediction_indices:
            for gt_index in unmatched_gt_indices:
                current_iou = float(
                    iou_matrix[prediction_index, gt_index]
                )

                if current_iou > best_iou:
                    best_iou = current_iou
                    best_prediction_index = prediction_index
                    best_gt_index = gt_index

        if (
            best_prediction_index is None
            or best_gt_index is None
        ):
            break

        matches.append(
            {
                "prediction_index": best_prediction_index,
                "gt_index": best_gt_index,
                "iou": best_iou,
            }
        )

        unmatched_prediction_indices.remove(
            best_prediction_index
        )
        unmatched_gt_indices.remove(best_gt_index)

    # Unmatched GT instances retain IoU 0.
    gt_ious = np.zeros(num_gt, dtype=np.float32)

    for match in matches:
        gt_ious[match["gt_index"]] = match["iou"]

    return {
        "matches": matches,
        "gt_ious": gt_ious,
        "mean_gt_iou": float(gt_ious.mean()),
        "minimum_gt_iou": float(gt_ious.min()),
        "num_gt": num_gt,
        "num_predictions": num_predictions,
        "num_unmatched_gt": len(unmatched_gt_indices),
        "num_unmatched_predictions": len(
            unmatched_prediction_indices
        ),
        "unmatched_gt_indices": sorted(
            unmatched_gt_indices
        ),
        "unmatched_prediction_indices": sorted(
            unmatched_prediction_indices
        ),
    }

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Patch, Rectangle


def draw_per_class_result(
    axis: plt.Axes,
    record: dict,
) -> None:
    """
    Draw the GT and prediction boxes for one selected class.

    Solid boxes  = ground truth
    Dashed boxes = predictions
    """
    image = cv2.imread(str(record["image_path"]))

    if image is None:
        raise RuntimeError(
            f"Could not read image: {record['image_path']}"
        )

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image_height = image.shape[0]

    axis.imshow(image)

    # Draw GT boxes with solid lines.
    for gt_index, box in enumerate(record["gt_boxes"]):
        x1, y1, x2, y2 = box
        gt_iou = record["gt_ious"][gt_index]

        axis.add_patch(
            Rectangle(
                xy=(x1, y1),
                width=x2 - x1,
                height=y2 - y1,
                fill=False,
                linewidth=2.5,
                linestyle="-",
                color="green",
            )
        )

        axis.text(
            x1,
            max(0, y1 - 5),
            f"GT {record['class_name']} | IoU={gt_iou:.2f}",
            fontsize=8,
            bbox={
                "facecolor": "white",
                "alpha": 0.75,
                "pad": 1,
            },
        )

    # Draw same-class prediction boxes with dashed lines.
    for box, confidence in zip(
        record["pred_boxes"],
        record["pred_confidences"],
    ):
        x1, y1, x2, y2 = box

        axis.add_patch(
            Rectangle(
                xy=(x1, y1),
                width=x2 - x1,
                height=y2 - y1,
                fill=False,
                linewidth=2.5,
                linestyle="--",
                color="blue",
            )
        )

        axis.text(
            x1,
            min(image_height - 2, y2 + 14),
            f"Pred {record['class_name']} | {confidence:.2f}",
            fontsize=8,
            bbox={
                "facecolor": "white",
                "alpha": 0.75,
                "pad": 1,
            },
        )

    axis.set_title(
        f"{record['image_path'].name}\n"
        f"mean GT IoU={record['mean_gt_iou']:.3f} | "
        f"min IoU={record['minimum_gt_iou']:.3f}\n"
        f"GT={record['num_gt']} | "
        f"pred={record['num_predictions']} | "
        f"missed GT={record['num_unmatched_gt']} | "
        f"extra pred={record['num_unmatched_predictions']}",
        fontsize=9,
    )

    axis.axis("off")


def visualize_records_for_class(
    class_id: int,
    class_name: str,
    records: list[dict],
    output_directory: Path | None = None,
    name_suffix: str | None = None,
) -> None:
    figure, axes = plt.subplots(
        nrows=len(records),
        ncols=1,
        figsize=(12, 24),
        squeeze=False,
    )

    for axis, record in zip(axes.flat, records):
        draw_per_class_result(axis, record)

        axis.text(
            0.01,
            0.99,
            name_suffix,
            transform=axis.transAxes,
            horizontalalignment="left",
            verticalalignment="top",
            fontsize=11,
            fontweight="bold",
            bbox={
                "facecolor": "white",
                "alpha": 0.85,
                "pad": 2,
            },
        )

    figure.suptitle(
        f"{class_name}: validation examples", fontsize=16,
    )

    figure.legend(
        handles=[
            Patch(
                facecolor="none",
                edgecolor="black",
                label="Ground truth: solid box",
            ),
            Patch(
                facecolor="none",
                edgecolor="black",
                label="Prediction: dashed box",
            ),
        ],
        loc="upper right",
    )

    figure.tight_layout(
        rect=(0.0, 0.0, 1.0, 0.95)
    )

    if output_directory is not None:
        output_directory.mkdir(
            parents=True,
            exist_ok=True,
        )

        safe_class_name = (
            class_name
            .replace(" ", "_")
            .replace("/", "_")
        )

        output_path = (
            output_directory
            / f"{class_id:02d}_{safe_class_name}_{name_suffix}.png"
        )

        figure.savefig(
            output_path,
            dpi=180,
            bbox_inches="tight",
        )

        print(f"Saved: {output_path}")

    plt.show()

In [ ]:
from ultralytics import YOLO
from collections import defaultdict
from pathlib import Path

import tqdm
import numpy as np

model = YOLO("/content/drive/MyDrive/DocuSketch/yolo26s_res640/weights/best.pt")


def read_image_manifest(
    dataset_root: Path,
    split: str = "val",
) -> list[Path]:
    """
    Read image paths from a YOLO manifest.

    Relative paths are resolved against dataset_root.
    """
    image_paths = []

    manifest_path = dataset_root / f"{split}.txt"

    with manifest_path.open("r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()

            if not line or line.startswith("#"):
                continue

            image_path = Path(line)

            if not image_path.is_absolute():
                image_path = dataset_root / image_path

            image_path = image_path.resolve()

            if not image_path.exists():
                raise FileNotFoundError(
                    f"Image does not exist: {image_path}"
                )

            image_paths.append(image_path)

    if not image_paths:
        raise RuntimeError(
            f"No image paths found in {manifest_path}"
        )

    return image_paths


image_paths = read_image_manifest(
    dataset_root=path_root,
    split="val",
)

class_names = {
    int(class_id): class_name
    for class_id, class_name in model.names.items()
}

# One list per class. Each list contains one record for every
# validation image containing at least one GT instance of that class.
per_class_records = defaultdict(list)

prediction_results = model.predict(
    source=[str(path) for path in image_paths],
    imgsz=640,
    conf=0.25,
    stream=True,
    verbose=False,
)

tqdm_iterator = tqdm.tqdm(zip(image_paths, prediction_results), total=len(image_paths))

for image_path, result in tqdm_iterator:
    tqdm_iterator.set_description(f"Processing {image_path.name}")

    gt_boxes, gt_classes = load_yolo_ground_truth(
        image_path
    )

    if result.boxes is None or len(result.boxes) == 0:
        pred_boxes = np.empty(
            (0, 4),
            dtype=np.float32,
        )
        pred_classes = np.empty(
            (0,),
            dtype=np.int64,
        )
        pred_confidences = np.empty(
            (0,),
            dtype=np.float32,
        )
    else:
        pred_boxes = (
            result.boxes.xyxy
            .detach()
            .cpu()
            .numpy()
            .astype(np.float32)
        )

        pred_classes = (
            result.boxes.cls
            .detach()
            .cpu()
            .numpy()
            .astype(np.int64)
        )

        pred_confidences = (
            result.boxes.conf
            .detach()
            .cpu()
            .numpy()
            .astype(np.float32)
        )

    # Evaluate only classes represented by GT in this image.
    for class_id in np.unique(gt_classes):
        class_id = int(class_id)

        gt_mask = gt_classes == class_id
        pred_mask = pred_classes == class_id

        class_gt_boxes = gt_boxes[gt_mask]
        class_pred_boxes = pred_boxes[pred_mask]
        class_pred_confidences = pred_confidences[pred_mask]

        evaluation = match_boxes_by_iou(
            pred_boxes=class_pred_boxes,
            gt_boxes=class_gt_boxes,
        )

        record = {
            "image_path": image_path,
            "class_id": class_id,
            "class_name": class_names[class_id],
            "gt_boxes": class_gt_boxes,
            "pred_boxes": class_pred_boxes,
            "pred_confidences": class_pred_confidences,
            **evaluation,
        }

        per_class_records[class_id].append(record)

print(
    f"Finished processing {len(image_paths)} validation images."
)

for class_id in sorted(per_class_records):
    print(
        f"{class_names[class_id]:20s}: "
        f"{len(per_class_records[class_id])} images"
    )

In [ ]:
from pathlib import Path

best_and_worst_by_class = {}

viz_count = 4

for class_id in sorted(per_class_records):
    class_records = per_class_records[class_id]

    if len(class_records) < viz_count + viz_count:
        print(
            f"Skipping {class_names[class_id]}: "
            f"only {len(class_records)} eligible images."
        )
        continue

    sorted_records = sorted(
        class_records,
        key=lambda record: (
            record["mean_gt_iou"],
            record["minimum_gt_iou"],
            -record["num_unmatched_gt"],
            -record["num_unmatched_predictions"],
        ),
    )

    worst_records = sorted_records[:viz_count]

    best_records = list(
        reversed(sorted_records[-viz_count:])
    )

    best_and_worst_by_class[class_id] = {
        "best": best_records,
        "worst": worst_records,
    }

    print(f"\nClass: {class_names[class_id]}")

    print("  Best:")
    for record in best_records:
        print(
            f"    {record['image_path'].name:30s} "
            f"mean IoU={record['mean_gt_iou']:.3f}, "
            f"min IoU={record['minimum_gt_iou']:.3f}, "
            f"GT={record['num_gt']}, "
            f"pred={record['num_predictions']}"
        )

    print("  Worst:")
    for record in worst_records:
        print(
            f"    {record['image_path'].name:30s} "
            f"mean IoU={record['mean_gt_iou']:.3f}, "
            f"min IoU={record['minimum_gt_iou']:.3f}, "
            f"GT={record['num_gt']}, "
            f"pred={record['num_predictions']}, "
            f"missed={record['num_unmatched_gt']}, "
            f"extra={record['num_unmatched_predictions']}"
        )

    visualize_records_for_class(
        class_id=class_id,
        class_name=class_names[class_id],
        records=best_records,
        output_directory=Path("/content/drive/MyDrive/DocuSketch/results_per_class"),
        name_suffix="best"
    )

    visualize_records_for_class(
        class_id=class_id,
        class_name=class_names[class_id],
        records=worst_records,
        output_directory=Path("/content/drive/MyDrive/DocuSketch/results_per_class"),
        name_suffix="worst"
    )